# PageXML line visualiser

Colours every text line of a PageXML over its page image. Five shapes are
available per line:

1. **Coords polygon** — the line's own `Coords` filled from its upper edge to
   its lower one. Where an exporter writes polygonal coordinates these already
   follow the words, rising over tall letters and dropping between them, so
   nothing needs computing. This is the default.
2. **Ink contour** — a band measured from the page image, for exports whose
   `Coords` are plain quadrilaterals and so cannot follow anything.
3. **Baseline band** — a band offset from the `Baseline`, undulating with the
   line but of even height.
4. **Straight** — a level rectangle per line.
5. **Letters** — the strokes themselves, coloured through an ink stencil.

**Files.** Point cell 5 at a folder (or an export zip) and every PageXML in it
is found and paired with its image automatically, using the `imageFilename`
attribute each PageXML declares. Pages are then chosen from a dropdown.

## 1. Imports

In [72]:
import colorsys, copy, math, re, time, zipfile
import xml.etree.ElementTree as ET
from pathlib import Path

import numpy as np
from numpy.lib.stride_tricks import sliding_window_view
from PIL import Image, ImageChops, ImageDraw, ImageFilter
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display

state = {'regions': None, 'image': None, 'gray': None, 'image_name': None,
         'colours': {}, 'pages': [], 'out': widgets.Output(),
         'colour_box': widgets.VBox(), 'panel_built': False}

print('Imports fine.')

Imports fine.


## 2. Parsing

In [73]:
NS = {'pc': 'http://schema.primaresearch.org/PAGE/gts/pagecontent/2013-07-15'}


def _find(el, tag):
    found = el.find(f'pc:{tag}', NS)
    return el.find(tag) if found is None else found


def parse_points(s):
    return [(int(float(x)), int(float(y)))
            for x, y in (p.split(',') for p in (s or '').split())]


def region_type(region):
    m = re.search(r'type\s*:\s*([A-Za-z-]+)', region.get('custom', '') or '')
    return m.group(1) if m else 'paragraph'


def parse_pagexml(xml_bytes):
    """Returns (regions, declared_page_size). The size lets us catch a scan
    that has been resampled since the transcription was made."""
    root = ET.fromstring(xml_bytes)
    page = _find(root, 'Page')
    try:
        page_size = (int(page.get('imageWidth')), int(page.get('imageHeight')))
    except (AttributeError, TypeError, ValueError):
        page_size = None
    regions = []
    found = root.findall('.//pc:TextRegion', NS) or root.findall('.//TextRegion')

    for idx, reg in enumerate(found):
        rc_el = _find(reg, 'Coords')
        lines = []
        for ln in (reg.findall('pc:TextLine', NS) or reg.findall('TextLine')):
            c_el, b_el, te = _find(ln, 'Coords'), _find(ln, 'Baseline'), _find(ln, 'TextEquiv')
            text = ''
            if te is not None:
                u = _find(te, 'Unicode')
                if u is not None and u.text:
                    text = u.text
            lines.append({
                'coords': parse_points(c_el.get('points')) if c_el is not None else [],
                'baseline': parse_points(b_el.get('points')) if b_el is not None else [],
                'text': text})

        regions.append({'id': reg.get('id', f'region_{idx}'),
                        'type': region_type(reg),
                        'coords': parse_points(rc_el.get('points')) if rc_el is not None else [],
                        'lines': lines})
    return regions, page_size


def scale_regions(regions, factor):
    """Rescale every coordinate, for images resampled since transcription."""
    fx, fy = factor
    for reg in regions:
        reg['coords'] = [(x * fx, y * fy) for x, y in reg['coords']]
        for ln in reg['lines']:
            ln['coords'] = [(x * fx, y * fy) for x, y in ln['coords']]
            ln['baseline'] = [(x * fx, y * fy) for x, y in ln['baseline']]
    return regions


print('Parser ready.')

Parser ready.


## 3. Geometry

`ink_contour` reads the page image. For every column of a line it finds the
highest and lowest dark pixel within a window around the baseline, widens that
profile so an isolated ascender becomes a lobe rather than a spike, smooths it,
and pads it away from the ink. `limit_up` and `limit_down` keep a band out of
its neighbours.

In [74]:
def line_height(line, fallback=40):
    if line['coords']:
        ys = [p[1] for p in line['coords']]
        if max(ys) - min(ys) > 2:
            return max(ys) - min(ys)
    return fallback


def baseline_y_at(baseline, xs):
    bx = np.array([p[0] for p in baseline], dtype=float)
    by = np.array([p[1] for p in baseline], dtype=float)
    o = np.argsort(bx)
    return np.interp(xs, bx[o], by[o])


def _roll(a, w, how):
    """Rolling min / max / mean with edge padding."""
    w = max(1, int(w) | 1)
    win = sliding_window_view(np.pad(a, w // 2, mode='edge'), w)
    return {'min': win.min, 'max': win.max, 'mean': win.mean}[how](axis=-1)


def densify(points, step=12):
    if len(points) < 2:
        return list(points)
    out = []
    for (x0, y0), (x1, y1) in zip(points, points[1:]):
        n = max(1, int(math.hypot(x1 - x0, y1 - y0) / step))
        out += [(x0 + (x1 - x0) * i / n, y0 + (y1 - y0) * i / n) for i in range(n)]
    out.append(points[-1])
    return out


def baseline_band(baseline, h, ascender, descender, step=10):
    pts = densify(baseline, step)
    return ([(x, y - h * ascender) for x, y in pts],
            [(x, y + h * descender) for x, y in pts])


def straight_band(line, h, ascender, descender):
    """A level rectangle for the line, ignoring the writing's undulation."""
    pts = line['coords'] or line['baseline']
    if not pts:
        return None
    xs = [p[0] for p in pts]
    x0, x1 = min(xs), max(xs)
    if line['coords']:
        ys = [p[1] for p in line['coords']]
        y0, y1 = min(ys), max(ys)
    else:
        y = float(np.mean([p[1] for p in line['baseline']]))
        y0, y1 = y - h * ascender, y + h * descender
    return ([(x0, y0), (x1, y0)], [(x0, y1), (x1, y1)])


def _ink_field(gray, baseline, h, sensitivity=1.0, max_asc=1.15, max_desc=0.65,
               max_gap=0.40, limit_up=None, limit_down=None):
    """Locate the ink of one line. Shared by the contour and the letter fill.

    Two safeguards keep the result on the writing. The threshold is taken
    from a core band around the baseline, so a black scanner background or
    binding shadow inside the window cannot drag it. And ink counts only
    where it is reachable from the baseline through further ink, tolerating
    gaps shorter than `max_gap` of the line height: dark matter separated
    from the line by clear parchment is ignored rather than swallowed.
    """
    H, W = gray.shape
    xs_all = [p[0] for p in baseline]
    x0, x1 = int(max(0, min(xs_all))), int(min(W - 1, max(xs_all)))
    if x1 - x0 < 4:
        return None

    xs = np.arange(x0, x1 + 1)
    by = baseline_y_at(baseline, xs)

    up = max_asc * h if limit_up is None else min(max_asc * h, limit_up)
    down = max_desc * h if limit_down is None else min(max_desc * h, limit_down)

    lo = int(np.clip(by.min() - up, 0, H - 1))
    hi = int(np.clip(by.max() + down, 0, H - 1))
    if hi - lo < 3:
        return None

    strip = gray[lo:hi + 1, x0:x1 + 1].astype(np.float32)
    R, C = strip.shape
    rows = np.arange(lo, hi + 1)[:, None]

    core = np.abs(rows - by[None, :]) <= 0.6 * h
    sample = strip[core] if core.sum() > 50 else strip
    thr = float(sample.mean() - sensitivity * 0.55 * sample.std())

    ink = strip < thr
    ink &= (rows >= np.maximum(by - up, lo)[None, :])
    ink &= (rows <= np.minimum(by + down, hi)[None, :])

    g = max(3, int(max_gap * h)) | 1
    closed = sliding_window_view(np.pad(ink, ((g // 2, g // 2), (0, 0))),
                                 g, axis=0).max(axis=-1)

    b_idx = np.clip((by - lo).astype(int), 0, R - 1)
    cols = np.arange(C)

    def reach(limit_px, sign):
        k = np.arange(max(2, int(limit_px) + 1))
        idx = b_idx[None, :] + sign * k[:, None]
        ok = (idx >= 0) & (idx <= R - 1)
        got = closed[np.clip(idx, 0, R - 1), cols[None, :]] & ok
        return np.cumprod(got, axis=0).astype(bool).sum(axis=0)

    n_up, n_down = reach(up, -1), reach(down, +1)
    return {'xs': xs, 'by': by, 'lo': lo, 'x0': x0, 'R': R, 'C': C,
            'ink': ink, 'b_idx': b_idx, 'n_up': n_up, 'n_down': n_down,
            'up': up, 'down': down, 'H': H}


def ink_contour(gray, baseline, h, sensitivity=1.0, max_asc=1.15, max_desc=0.65,
                pad_frac=0.16, smooth_frac=0.55, grow_frac=0.35, step=6,
                max_gap=0.40, hug=0.0, limit_up=None, limit_down=None):
    """Top and bottom edges of a band that hugs the ink around one baseline."""
    f = _ink_field(gray, baseline, h, sensitivity, max_asc, max_desc,
                   max_gap, limit_up, limit_down)
    if f is None:
        return None

    xs, by, lo, b_idx = f['xs'], f['by'], f['lo'], f['b_idx']
    n_up, n_down, H = f['n_up'], f['n_down'], f['H']
    has = n_up + n_down > 0

    top = np.where(has, lo + b_idx - np.maximum(n_up - 1, 0), by - 0.35 * h)
    bot = np.where(has, lo + b_idx + np.maximum(n_down - 1, 0), by + 0.15 * h)

    # hug pulls the contour in towards the letters: less growing, less
    # smoothing, less padding, and a finer sampling step to catch detail
    grow_frac *= 1 - 0.80 * hug
    smooth_frac *= 1 - 0.75 * hug
    pad_frac *= 1 - 0.85 * hug
    step = max(2, int(round(step * (1 - 0.6 * hug))))

    gw, sw = max(3, int(grow_frac * h)), max(3, int(smooth_frac * h))
    top = _roll(_roll(top, gw, 'min'), sw, 'mean') - pad_frac * h
    bot = _roll(_roll(bot, gw, 'max'), sw, 'mean') + pad_frac * h

    floor_up, floor_down = 0.12 * h * (1 - 0.7 * hug), 0.06 * h * (1 - 0.7 * hug)
    top = np.clip(np.minimum(top, by - floor_up),
                  np.maximum(by - f['up'], 0), H - 1)
    bot = np.clip(np.maximum(bot, by + floor_down), 0,
                  np.minimum(by + f['down'], H - 1))

    sel = np.unique(np.r_[np.arange(0, len(xs), max(1, step)), len(xs) - 1])
    return ([(float(xs[i]), float(top[i])) for i in sel],
            [(float(xs[i]), float(bot[i])) for i in sel])


def ink_mask(gray, baseline, h, sensitivity=1.0, max_asc=1.15, max_desc=0.65,
             max_gap=0.40, thicken=0, limit_up=None, limit_down=None):
    """The letters themselves: a boolean mask of the line's ink, and where
    it sits. `thicken` grows the strokes by that many pixels, which keeps a
    fine hand visible once colour is laid over it."""
    f = _ink_field(gray, baseline, h, sensitivity, max_asc, max_desc,
                   max_gap, limit_up, limit_down)
    if f is None:
        return None

    rows = np.arange(f['R'])[:, None]
    lower = (f['b_idx'] - np.maximum(f['n_up'] - 1, 0))[None, :]
    upper = (f['b_idx'] + np.maximum(f['n_down'] - 1, 0))[None, :]
    mask = f['ink'] & (rows >= lower) & (rows <= upper)
    if not mask.any():
        return None

    img = Image.fromarray((mask * 255).astype(np.uint8))
    if thicken > 0:
        img = img.filter(ImageFilter.MaxFilter(int(thicken) * 2 + 1))
    return img, f['x0'], f['lo']


def neighbour_limits(regions, share=0.85, min_overlap=0.25):
    """How far each line may reach before it meets another line.

    Every line on the page is considered, not merely its region-mates: a
    heading and the paragraph beneath it belong to different regions but
    still compete for the same interlinear space. Only lines that actually
    share horizontal extent constrain one another, so a marginale off in
    the margin does not clamp a text line it never touches.

    The share is deliberately generous: in a cursive hand the ascenders of
    one line genuinely reach into the space above, and suppressing that
    flattens the contour into a strip. Bands may therefore overlap; what
    stops one covering another is the first-claim rule in `render`.

    Keys are (region index, line index).
    """
    items = []
    for ri, reg in enumerate(regions):
        for li, line in enumerate(reg['lines']):
            bl = line['baseline']
            if len(bl) < 2:
                continue
            xs = [p[0] for p in bl]
            items.append((ri, li, min(xs), max(xs),
                          float(np.mean([p[1] for p in bl]))))

    lim = {}
    for ri, li, x0, x1, y in items:
        up = down = None
        for rj, lj, X0, X1, Y in items:
            if rj == ri and lj == li:
                continue
            overlap = min(x1, X1) - max(x0, X0)
            if overlap <= min_overlap * max(1.0, min(x1 - x0, X1 - X0)):
                continue
            if Y < y:
                up = y - Y if up is None else min(up, y - Y)
            elif Y > y:
                down = Y - y if down is None else min(down, Y - y)
        lim[(ri, li)] = (None if up is None else up * share,
                         None if down is None else down * share)
    return lim


print('Geometry ready.')

Geometry ready.


## 4. Colour and rendering

In [75]:
PALETTE = ['#E85D75', '#7C6BA3', '#4A9FB5', '#A89B5B', '#C96B4D',
           '#6FB88F', '#9B7E8F', '#C77DA6', '#B8956B', '#5B8FA8']


def hex_to_rgb(c):
    c = c.lstrip('#')
    return tuple(int(c[i:i + 2], 16) for i in (0, 2, 4))


def pearl_shades(rgb, n, amount, phase=0.0):
    """n hue- and lightness-shifted variants, sweeping like nacre."""
    h, l, s = colorsys.rgb_to_hls(*[v / 255 for v in rgb])
    out = []
    for i in range(max(n, 1)):
        t = i / max(n - 1, 1)
        w1 = math.sin(2 * math.pi * 1.15 * t + phase)
        w2 = math.sin(2 * math.pi * 2.35 * t + phase * 1.7)
        rr, gg, bb = colorsys.hls_to_rgb(
            (h + 0.045 * amount * w1) % 1.0,
            min(0.92, max(0.18, l + 0.13 * amount * w2)),
            min(1.0, max(0.15, s * (1 - 0.18 * amount * w1))))
        out.append((int(rr * 255), int(gg * 255), int(bb * 255)))
    return out


def render(image, gray, regions, colours, opacity=0.45, pearl=0.5,
           mode='coords', show_boxes=True, box_width=3,
           ascender=0.95, descender=0.28,
           sensitivity=1.0, max_asc=1.15, max_desc=0.65,
           padding=0.16, smoothing=0.55, crowd=0.55, hug=0.0, thicken=1):
    """Composite coloured line bands over the page image.

    Opacity is applied through a mask, so overlapping bands never compound
    into darker patches.
    """
    base = image.convert('RGB')
    fill_col = Image.new('RGB', base.size, (255, 255, 255))
    fill_msk = Image.new('L', base.size, 0)
    claimed = Image.new('L', base.size, 0)
    edge_col = Image.new('RGB', base.size, (255, 255, 255))
    edge_msk = Image.new('L', base.size, 0)
    de, dem = ImageDraw.Draw(edge_col), ImageDraw.Draw(edge_msk)

    def paint(box, patch, shape):
        """Paint one line, but only onto pixels no earlier line took."""
        free = ImageChops.subtract(shape, claimed.crop(box))
        fill_col.paste(patch, box, free)
        fill_msk.paste(255, box, free)
        claimed.paste(255, box, shape)

    def claim_polygon(points, shades):
        """Fill a Coords polygon outright, from its upper to its lower edge."""
        xs = [p[0] for p in points]
        ys = [p[1] for p in points]
        x0, y0 = max(0, int(min(xs)) - 1), max(0, int(min(ys)) - 1)
        x1 = min(base.width, int(max(xs)) + 2)
        y1 = min(base.height, int(max(ys)) + 2)
        if x1 <= x0 or y1 <= y0:
            return
        box = (x0, y0, x1, y1)

        shape = Image.new('L', (x1 - x0, y1 - y0), 0)
        ImageDraw.Draw(shape).polygon([(x - x0, y - y0) for x, y in points],
                                      fill=255)
        patch = Image.new('RGB', shape.size, shades[0])
        if len(shades) > 1:                      # sweep the hue along the line
            ramp = Image.new('RGB', (len(shades), 1))
            ramp.putdata(shades)
            patch = ramp.resize(shape.size, Image.BILINEAR)
        paint(box, patch, shape)

    def claim_mask(stamp, x0, y0, shades):
        """Colour the letters themselves, through the ink mask."""
        w, hgt = stamp.size
        box = (x0, y0, min(base.width, x0 + w), min(base.height, y0 + hgt))
        if box[2] <= box[0] or box[3] <= box[1]:
            return
        stamp = stamp.crop((0, 0, box[2] - box[0], box[3] - box[1]))
        patch = Image.new('RGB', stamp.size, shades[0])
        if len(shades) > 1:                      # sweep the hue along the line
            ramp = Image.new('RGB', (len(shades), 1))
            ramp.putdata(shades)
            patch = ramp.resize(stamp.size, Image.BILINEAR)
        paint(box, patch, stamp)

    def claim(quads, shades):
        """Paint one line's band, but only onto pixels no earlier band took.

        Bands are allowed to overlap so each can follow its own ascenders;
        this keeps the overlap from hiding whichever line came first.
        """
        xs = [p[0] for q in quads for p in q]
        ys = [p[1] for q in quads for p in q]
        x0, y0 = max(0, int(min(xs)) - 2), max(0, int(min(ys)) - 2)
        x1 = min(base.width, int(max(xs)) + 3)
        y1 = min(base.height, int(max(ys)) + 3)
        if x1 <= x0 or y1 <= y0:
            return
        box = (x0, y0, x1, y1)

        patch = Image.new('RGB', (x1 - x0, y1 - y0), (255, 255, 255))
        shape = Image.new('L', (x1 - x0, y1 - y0), 0)
        dp, ds = ImageDraw.Draw(patch), ImageDraw.Draw(shape)
        for quad, colour in zip(quads, shades):
            moved = [(x - x0, y - y0) for x, y in quad]
            dp.polygon(moved, fill=colour)
            ds.polygon(moved, fill=255)

        paint(box, patch, shape)

    limits = (neighbour_limits(regions, crowd)
              if mode in ('ink', 'letters') else {})

    for ridx, reg in enumerate(regions):
        rgb = hex_to_rgb(colours.get(reg['id'], '#E85D75'))

        for lidx, line in enumerate(reg['lines']):
            bl, h = line['baseline'], line_height(line)
            edges = None

            if mode == 'coords':
                if len(line['coords']) >= 3:
                    shades = (pearl_shades(rgb, 24, pearl) if pearl > 0.01
                              else [rgb])
                    claim_polygon(line['coords'], shades)
                    if show_boxes and line['coords']:
                        de.polygon(line['coords'], outline=rgb, width=box_width)
                        dem.polygon(line['coords'], outline=255, width=box_width)
                continue

            if mode == 'letters' and len(bl) >= 2 and gray is not None:
                up, down = limits.get((ridx, lidx), (None, None))
                got = ink_mask(gray, bl, h, sensitivity, max_asc, max_desc,
                               thicken=thicken, limit_up=up, limit_down=down)
                if got is not None:
                    stamp, mx, my = got
                    shades = (pearl_shades(rgb, max(2, stamp.width // 6), pearl)
                              if pearl > 0.01 else [rgb])
                    claim_mask(stamp, mx, my, shades)
                    if show_boxes and line['coords']:
                        de.polygon(line['coords'], outline=rgb, width=box_width)
                        dem.polygon(line['coords'], outline=255, width=box_width)
                    continue

            if mode == 'straight':
                edges = straight_band(line, h, ascender, descender)
            elif mode == 'ink' and len(bl) >= 2 and gray is not None:
                up, down = limits.get((ridx, lidx), (None, None))
                edges = ink_contour(gray, bl, h, sensitivity, max_asc, max_desc,
                                    padding, smoothing, hug=hug,
                                    limit_up=up, limit_down=down)
            if edges is None and mode != 'straight' and len(bl) >= 2:
                edges = baseline_band(bl, h, ascender, descender)
            if edges is None:
                edges = straight_band(line, h, ascender, descender)

            if edges is not None:
                top, bot = edges
                if pearl > 0.01 and len(top) > 2:
                    shades = pearl_shades(rgb, len(top) - 1, pearl,
                                          phase=ridx * 1.9 + lidx * 0.7)
                    quads = [[bot[i], bot[i + 1], top[i + 1], top[i]]
                             for i in range(len(top) - 1)]
                    claim(quads, shades)
                else:
                    claim([bot + top[::-1]], [rgb])


            if show_boxes and line['coords']:
                de.polygon(line['coords'], outline=rgb, width=box_width)
                dem.polygon(line['coords'], outline=255, width=box_width)

    out = base.copy()
    out.paste(fill_col, mask=fill_msk.point(lambda v: int(v * opacity)))
    out.paste(edge_col, mask=edge_msk)
    return out


print('Renderer ready.')

Renderer ready.


## 5. Find your pages

Set `FOLDER` to whichever directory holds the material — the export folder, the
`page` subfolder, or the folder the images sit in; the search looks inside it,
in its parent, and in sibling folders called `images`, `scans` and the like.
Set `ZIP` instead to point straight at an export zip, which is unpacked beside
itself. `IMAGE_FOLDER` is only needed if the scans live somewhere unrelated.

Pairing uses the `imageFilename` each PageXML declares, falling back to
matching by filename. Anything left unpaired is reported rather than guessed at.

In [76]:
# Either a folder or a zip works here, and it does not matter which
# variable you use; on Windows keep the r prefix so backslashes survive.
FOLDER = r'.'          # e.g. r'C:\Users\you\Documents\pagexml'
ZIP = None             # e.g. r'export_job_30342296.zip'
IMAGE_FOLDER = None    # only if the scans live somewhere unrelated

IMG_EXT = ('.jpg', '.jpeg', '.png', '.tif', '.tiff', '.jp2', '.bmp', '.webp')
SIBLINGS = ('images', 'Images', 'img', 'Img', 'jpg', 'JPG', 'scans', 'Scans')


def unpack(zip_path, target=None):
    """Extract an export zip, ignoring any absolute paths inside it."""
    zip_path = Path(zip_path)
    # beside the notebook, not beside the zip, which may be read-only
    target = Path(target or Path.cwd() / zip_path.stem)
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as z:
        for m in z.infolist():
            name = m.filename.lstrip('/')
            if not name or name.endswith('/'):
                continue
            dest = target / name
            dest.parent.mkdir(parents=True, exist_ok=True)
            with z.open(m) as src, open(dest, 'wb') as out:
                out.write(src.read())
    return target


def image_filename_in(xml_path):
    head = Path(xml_path).read_text(encoding='utf-8', errors='ignore')[:6000]
    m = re.search(r'imageFilename\s*=\s*"([^"]+)"', head)
    return Path(m.group(1)).name if m else None


def index_images(roots):
    """Only folders we expect images in are walked recursively, so a stray
    scan elsewhere can never be paired with a page by accident."""
    by_name, by_stem = {}, {}
    for root, recursive in roots:
        root = Path(root)
        if not root.is_dir():
            continue
        for p in (root.rglob('*') if recursive else root.glob('*')):
            if p.suffix.lower() in IMG_EXT and p.is_file():
                by_name.setdefault(p.name.lower(), p)
                by_stem.setdefault(p.stem.lower(), p)
    return by_name, by_stem


def find_pairs(xml_root, image_root=None):
    xml_root = Path(xml_root)
    xmls = sorted(p for p in xml_root.rglob('*.xml')
                  if p.name.lower() not in ('mets.xml', 'metadata.xml'))

    roots = [(xml_root, True)]
    if image_root:
        roots.insert(0, (Path(image_root), True))
    for up in (xml_root.parent, xml_root.parent.parent):
        roots.append((up, False))
        roots += [(up / s, True) for s in SIBLINGS]
    by_name, by_stem = index_images(roots)

    pairs = []
    for x in xmls:
        want = image_filename_in(x)
        img = None
        if want:
            img = by_name.get(want.lower()) or by_stem.get(Path(want).stem.lower())
        if img is None:
            img = by_stem.get(x.stem.lower())
        if img is None:
            img = by_stem.get(re.sub(r'^\d+[_-]', '', x.stem).lower())
        pairs.append({'xml': x, 'image': img, 'declared': want})
    return pairs


def resolve_root(folder, zip_path):
    """Accept a folder or a zip in either variable, and say plainly when a
    path does not exist rather than failing deeper down."""
    for candidate in (zip_path, folder):
        if not candidate:
            continue
        p = Path(candidate).expanduser()
        if p.is_dir():
            return p
        if p.is_file() and zipfile.is_zipfile(p):
            return unpack(p)
        raise FileNotFoundError(
            f'{p.resolve()} is neither a folder nor a zip. On Windows write the '
            f'path as r\'C:\\...\' and include the drive letter.')
    raise ValueError('Set FOLDER (or ZIP) at the top of this cell.')


root = resolve_root(FOLDER, ZIP)
state['pages'] = find_pairs(root, IMAGE_FOLDER)

ok = [p for p in state['pages'] if p['image']]
print(f'{len(state["pages"])} PageXML files under {root.resolve()}, {len(ok)} paired')
for p in state['pages']:
    print(f'  {p["xml"].name:<45} {p["image"].name if p["image"] else "no image found"}')
if not ok:
    print('\nNothing paired. Set IMAGE_FOLDER to wherever the scans are.')


def load_page(pair):
    regions, declared = parse_pagexml(pair['xml'].read_bytes())
    image = Image.open(pair['image'])

    if declared and declared != image.size:
        fx, fy = image.width / declared[0], image.height / declared[1]
        regions = scale_regions(regions, (fx, fy))
        print(f'  note: image is {image.width} x {image.height} but the XML was '
              f'written for {declared[0]} x {declared[1]}; coordinates rescaled '
              f'by {fx:.3f} x {fy:.3f}')

    state['regions'] = regions
    state['image'] = image
    state['gray'] = np.array(image.convert('L'))
    state['image_name'] = pair['image'].name
    state['colours'] = {r['id']: PALETTE[i % len(PALETTE)]
                        for i, r in enumerate(state['regions'])}
    n = sum(len(r['lines']) for r in state['regions'])
    print(f'{pair["xml"].name}: {len(state["regions"])} regions, {n} lines, '
          f'{state["image"].width} x {state["image"].height}')


if ok:
    load_page(ok[0])

4 PageXML files under C:\Users\RomeinCA\OneDrive - University of Twente\Publications_en_Presentaties\Melissa, 4 paired
  0001_100989379.2.xml                          100989379.2.jpg
  0002_107398037.2.xml                          107398037.2.jpg
  0003_107955063.2.xml                          107955063.2.jpg
  0004_107955481.2.xml                          107955481.2.jpg
0001_100989379.2.xml: 2 regions, 13 lines, 2500 x 2890


## 6. Controls and preview

Switching page in the dropdown reloads it and redraws, keeping your settings.
`Ink sensitivity` raises or lowers the darkness threshold: raise it on a clean
scan, lower it where the parchment is stained or the ink faded. `Crowding`
caps how far a band may reach towards its neighbours. `Pearl` at 0 gives flat
colour.

In [77]:
page_w = widgets.Dropdown(
    options=[(p['xml'].name, i) for i, p in enumerate(state['pages']) if p['image']],
    description='Page', layout=widgets.Layout(width='520px'))

mode_w = widgets.ToggleButtons(
    options=[('Coords polygon', 'coords'), ('Ink contour', 'ink'),
             ('Baseline band', 'baseline'), ('Straight', 'straight'),
             ('Letters', 'letters')],
    value='coords', description='Shape',
    layout=widgets.Layout(width='auto'))
hug_w = widgets.FloatSlider(value=0.0, min=0.0, max=1.0, step=0.05,
                            description='Hug', continuous_update=False)
thicken_w = widgets.IntSlider(value=1, min=0, max=6, description='Thicken',
                              continuous_update=False)
opacity_w = widgets.FloatSlider(value=0.45, min=0.05, max=1.0, step=0.05,
                                description='Opacity', continuous_update=False)
pearl_w = widgets.FloatSlider(value=0.5, min=0.0, max=1.0, step=0.05,
                              description='Pearl', continuous_update=False)
sens_w = widgets.FloatSlider(value=1.0, min=0.2, max=2.5, step=0.1,
                             description='Ink sens.', continuous_update=False)
pad_w = widgets.FloatSlider(value=0.16, min=0.0, max=0.6, step=0.02,
                            description='Padding', continuous_update=False)
smooth_w = widgets.FloatSlider(value=0.55, min=0.1, max=2.0, step=0.05,
                               description='Smoothing', continuous_update=False)
crowd_w = widgets.FloatSlider(value=0.85, min=0.3, max=1.3, step=0.05,
                              description='Crowding', continuous_update=False)
maxasc_w = widgets.FloatSlider(value=1.15, min=0.5, max=2.2, step=0.05,
                               description='Max up', continuous_update=False)
maxdesc_w = widgets.FloatSlider(value=0.65, min=0.1, max=1.5, step=0.05,
                                description='Max down', continuous_update=False)
asc_w = widgets.FloatSlider(value=0.95, min=0.3, max=1.6, step=0.05,
                            description='Band up', continuous_update=False)
desc_w = widgets.FloatSlider(value=0.28, min=0.0, max=0.8, step=0.02,
                             description='Band down', continuous_update=False)
boxes_w = widgets.Checkbox(value=True, description='Coords outline')
boxwidth_w = widgets.IntSlider(value=3, min=1, max=10, description='Outline px')
bytype_w = widgets.Checkbox(value=False, description='Colour by type')
scale_w = widgets.IntSlider(value=1600, min=600, max=3000, step=100,
                            description='Preview px', continuous_update=False)


def effective_colours():
    if not bytype_w.value:
        return dict(state['colours'])
    types = sorted({r['type'] for r in state['regions']})
    by_type = {t: PALETTE[i % len(PALETTE)] for i, t in enumerate(types)}
    return {r['id']: by_type[r['type']] for r in state['regions']}


def prepared(width=None):
    """Image, grey channel and coordinates at a reduced width, cached.

    Previews are drawn at the size actually shown: a full-resolution page
    costs some seconds, the same page at preview width a fraction of one,
    and every slider move triggers a redraw.
    """
    if width is None or width >= state['image'].width:
        return state['image'], state['gray'], state['regions']
    key = (state['image_name'], int(width))
    if state.get('prep_key') != key:
        f = width / state['image'].width
        small = state['image'].resize((max(1, int(state['image'].width * f)),
                                       max(1, int(state['image'].height * f))))
        regions = scale_regions(copy.deepcopy(state['regions']), (f, f))
        state['prep_key'] = key
        state['prep'] = (small, np.array(small.convert('L')), regions)
    return state['prep']


def build_image(width=None):
    image, gray, regions = prepared(width)
    return render(image, gray, regions,
                  effective_colours(),
                  opacity=opacity_w.value, pearl=pearl_w.value,
                  mode=mode_w.value, show_boxes=boxes_w.value,
                  box_width=boxwidth_w.value,
                  ascender=asc_w.value, descender=desc_w.value,
                  sensitivity=sens_w.value, max_asc=maxasc_w.value,
                  max_desc=maxdesc_w.value, padding=pad_w.value,
                  smoothing=smooth_w.value, crowd=crowd_w.value,
                  hug=hug_w.value, thicken=thicken_w.value)


def refresh_colour_rows():
    rows = []
    for reg in state['regions']:
        picker = widgets.ColorPicker(value=state['colours'][reg['id']], concise=True)
        picker.observe(
            (lambda rid: lambda ch: state['colours'].__setitem__(rid, ch['new']))(reg['id']),
            names='value')
        rows.append(widgets.HBox([
            widgets.Label(f"{reg['id']}  ({reg['type']}, {len(reg['lines'])} lines)",
                          layout=widgets.Layout(width='330px')),
            picker]))
    state['colour_box'].children = tuple(rows)


def draw_preview(_=None):
    with state['out']:
        state['out'].clear_output(wait=True)
        t = time.time()
        prev = build_image(scale_w.value)
        fig, ax = plt.subplots(figsize=(14, 14 * prev.height / prev.width))
        ax.imshow(prev)
        ax.axis('off')
        plt.tight_layout()
        plt.show()
        print(f'{state["image_name"]} — preview {prev.width} x {prev.height} '
              f'of {state["image"].width} x {state["image"].height}, '
              f'{time.time() - t:.1f}s')


state.pop('prep_key', None)


def on_page_change(change):
    with state['out']:
        state['out'].clear_output(wait=True)
        print('loading…')
    load_page(state['pages'][change['new']])
    refresh_colour_rows()
    draw_preview()


page_w.observe(on_page_change, names='value')

preview_btn = widgets.Button(description='Preview', button_style='success')
preview_btn.on_click(draw_preview)

panel = widgets.VBox([
    widgets.HBox([page_w]),
    widgets.HBox([preview_btn, scale_w]),
    mode_w,
    widgets.HBox([widgets.VBox([opacity_w, pearl_w, hug_w, sens_w, pad_w]),
                  widgets.VBox([smooth_w, crowd_w, maxasc_w, maxdesc_w]),
                  widgets.VBox([asc_w, desc_w, thicken_w, boxes_w,
                                boxwidth_w, bytype_w])]),
    widgets.HTML('<b>Region colours</b> (ignored while <i>Colour by type</i> is ticked)'),
    state['colour_box']])

state['panel_built'] = True
refresh_colour_rows()
state['out'].clear_output()
display(panel, state['out'])
draw_preview()

Output()

## 7. Save

`Save PNG` writes the page on screen. `Save every page` runs the whole folder
with the settings currently set, which is where the automatic pairing earns
its keep.

In [78]:
OUT_DIR = Path('visualised')


def save(_=None):
    OUT_DIR.mkdir(exist_ok=True)
    print('rendering at full resolution…')
    img = build_image(None)
    name = OUT_DIR / f"{Path(state['image_name']).stem}_lines.png"
    img.save(name, compress_level=3)
    print(f'wrote {name}  ({img.width} x {img.height})')


def save_all(_=None):
    OUT_DIR.mkdir(exist_ok=True)
    keep = (state['regions'], state['image'], state['gray'],
            state['image_name'], state['colours'])
    for pair in [p for p in state['pages'] if p['image']]:
        load_page(pair)
        img = build_image(None)
        name = OUT_DIR / f"{pair['image'].stem}_lines.png"
        img.save(name, compress_level=3)
        print(f'  wrote {name}')
    (state['regions'], state['image'], state['gray'],
     state['image_name'], state['colours']) = keep
    print('done')


save_btn = widgets.Button(description='Save PNG', button_style='warning')
save_btn.on_click(save)
all_btn = widgets.Button(description='Save every page')
all_btn.on_click(save_all)
display(widgets.HBox([save_btn, all_btn]))